In [ ]:
agora esse:

from pyspark.sql.functions import *
from pyspark.sql.functions import col, when, lit, isnan, count, mean
from pyspark.sql.types import DateType
import seaborn as sns
import matplotlib.pyplot as plt

#Importa base com público e target
df_publico_target_spark = spark.createDataFrame(data_science.df_dummy_model_activation_dataprep)
%md
#1. Traz informações para features
df_publico_target_spark['dat_abertura_conta_AAAAMMDD'] = df_publico_target_spark['dat_abertura_conta'].dt.strftime('%Y%m%d')
#------------------------------
#------ Cadastro
#------------------------------

df_publico_target_spark.createOrReplaceTempView("df_publico_target_spark")

df_publico_target_spark_f = spark.sql(
    """
    SELECT a.*,
           b.dat_nascimento, b.vlr_salario_cliente, b.qtd_dependente, b.cod_estado_civil, b.nom_estado,
           c.qtd_hspn as hspn_venda
    FROM df_publico_target_spark AS a LEFT JOIN gold.dim_cartao_credito_cadastro AS b
          ON a.num_conta = b.num_conta
                                        LEFT JOIN gold.fato_cartao_credito_vendas AS c
          ON a.num_conta = c.num_conta
    """)

df_publico_target_spark_f.count()
#------------------------------
#------ Renda
#------------------------------

df_publico_target_spark_f.createOrReplaceTempView("df_publico_target_spark_f")

df_publico_target_spark_f = spark.sql(
    """
    SELECT a.*,
           b.flg_bf, coalesce(b.dat_referencia,0) as ref_renda, b.vlr_renda_risco, b.vlr_renda_serv_federal, b.vlr_mediana_bcvencer30, b.flg_regra_investidores, b.flg_regra_beneficio_cartao_ultra, b.flg_regra_prime, b.flg_regra_hospital_einstein, b.flg_regra_mei, b.flg_regra_motorista, b.flg_regra_entregador, b.vlr_mediana_pgtom6
    FROM df_publico_target_spark_f AS a LEFT JOIN diamond.ds_ren_pf_02_score AS b
          ON a.num_cpf_cnpj_cliente = b.num_cpf

    """)

df_publico_target_spark_f.createOrReplaceTempView("df_publico_target_spark_f")

df_publico_target_spark_f = spark.sql(
    """ select * from
        (select *,
                ROW_NUMBER() OVER (PARTITION BY num_conta ORDER BY ref_renda DESC) as rank
        from
            df_publico_target_spark_f)
    where rank=1
    """)

#dessa forma pega a renda mais recente, não necessariamente antes da abertura, mas é só uma base didática ...
df_publico_target_spark_f.count()
%md
#2. Função para criação de novas variáveis
#Função para criar variáveis
def adicionar_colunas(df):
    return df \
        .withColumn("idade", floor(datediff(col("dat_abertura_conta"), col("dat_nascimento")) / 365.25)) \
        .withColumn("limite_inicial/renda_risco", (col("vlr_limite_credito_inicial") / col("renda_risco"))) \
        .withColumn("limite_inicial/salario", (col("vlr_limite_credito_inicial") / col("vlr_salario_cliente"))) \
        .withColumn("renda_risco/salario", (col("renda_risco") / col("vlr_salario_cliente")))
df_publico_target_spark = adicionar_colunas(df_publico_target_spark)
%md
#3. Aplica regras de negócio
Os "imputers" para missing serão inclusos posteriormente, no pipeline na modelagem
# Limite: substituir pelo limite mínimo = 500 reais
df_publico_target_spark = df_publico_target_spark.withColumn(
    "vlr_limite_credito_inicial",
    when(col("vlr_limite_credito_inicial").isNull(), lit(500)).otherwise(col("vlr_limite_credito_inicial"))
)
# Categóricas: substitui nulos por 'desconhecido'
categorical_cols = ['cod_estado_civil', 'nom_estado', 'cod_canal_venda', 'cod_logo']

for col_name in categorical_cols:
    df_publico_target_spark = df_publico_target_spark.withColumn(
        col_name + "_filled",
        when((col(col_name).isNull()) | (col(col_name) == ""), lit("desconhecido")).otherwise(col(col_name))
    )
%md
**Análise Descritiva**
df_desc = df_publico_target_spark.toPandas()
# Seleciona colunas numéricas
num_cols = df_desc.select_dtypes(include=['float64', 'int64']).columns.tolist()

# Plota boxplot para cada uma
for col in num_cols:
    plt.figure(figsize=(10, 1))
    sns.boxplot(x=df_desc[col], color='lightblue')
    plt.title(f'Boxplot - {col}')
    plt.grid(True)
    plt.show()

# Seleciona colunas categóricas
cat_cols = df_desc.select_dtypes(include=['object']).columns.tolist()

# Plota frequência de cada uma
for col in cat_cols:
    plt.figure(figsize=(10, 4))
    df_desc[col].value_counts(dropna=False).plot(kind='bar')
    plt.title(f'Frequência - {col}')
    plt.xlabel(col)
    plt.ylabel('Contagem')
    plt.xticks(rotation=45)
    plt.grid(axis='y')
    plt.tight_layout()
    plt.show()

%md
#Salva base
df_publico_target_spark.write.mode("overwrite").saveAsTable("data_science.df_dummy_model_activation_feat")